## 1. 数据合并 
- `pd.merge()`: 类似于 SQL 的 join 操作，用于根据键列方向合并两个 DataFrame。
- `pd.concat()`: 用于沿轴（行或列）拼接多个 DataFrame。

In [ ]:
#导入Pandas库
import pandas as pd
#从相对路径下获取数据表
df1=pd.read_excel('https://oss.xinchanjiao.com/upload/default/20220914-1bbc5a0e-9409-4ffa-adab-06f15090e2fb.xlsx')
#显示数据前5行
df1.head()

In [ ]:
#读取Excel文件，利用converters参数强制设置“年"、“月"、"门店“三列转换为字符串类型。
df2=pd.read_excel(r'https://oss.xinchanjiao.com/upload/default/20220914-1bbc5a0e-9409-4ffa-adab-06f15090e2fb.xlsx',sheet_name='Sheet2')
#显示数据前5行
df2.head()

In [ ]:
#Sheet1表为主，左连接数据
#未指定列，以两个DataFrame列名的交集作为连接键
df3=pd.merge(df1,df2,how='left')
df3.head()

In [ ]:
#df1,df2用concat进行纵向连接
df4=pd.concat([df1,df2],sort=False)
df4

In [ ]:
#df1,df2，调整axis参数用concat进行横向列连接
df4=pd.concat([df1,df2],axis=1,sort=False)
df4

In [ ]:
#读取数据
df=pd.read_excel(r'https://oss.xinchanjiao.com/upload/default/20221109-780c231e-ec8d-4f7a-a878-15ca651452f7.xlsx')
#查找重复项
df.duplicated()

## 2. 数据清洗 (Data Cleaning)
- `duplicated()`: 查找重复行。
- `drop_duplicates()`: 删除重复行。
- `dropna()`: 删除包含缺失值 (NaN) 的行或列。
- `fillna()`: 填充缺失值。

In [ ]:
#删除重复项
df.drop_duplicates (inplace=True)
df.head()

In [ ]:
#删除重复项，并重置索引
df.drop_duplicates (inplace=True,ignore_index=True)
df

In [ ]:
#使用dropnaO在原数据上删除全为NaN的列
df.dropna(axis=1,how='all',inplace=True)
df

In [ ]:
#以O填充所有缺失值
df.fillna(0)

In [ ]:
#使用循环方式完成数据清洗
for i in range(df.shape[0]):
    for j in df.columns:
        df.loc[i, j] = str(df.loc[i, j]).replace(' ', '')
df

## 3. 数据转换与映射 (Transformation & Mapping)
- `applymap()`: 对 DataFrame 中的每一个元素应用函数（常用于字符串处理）。
- `map()`: 对 Series 中的每一个元素应用函数或字典映射。
- `apply()`: 沿轴（行或列）应用函数。

In [ ]:
#applymap函数完成数据清洗
df = df.applymap(lambda x:str(x).replace(' ',''))
df

In [ ]:
#将所有数字列转换为浮点数
df[['销售收入','销售成本','销售毛利']]=df[['销售收入','销售成本','销售毛利']].astype('float')
#以每列平均值填充该列缺失值，仅针对数值列
df.fillna(df.mean(numeric_only=True))

In [ ]:
#读取Excel文件
data=pd.read_excel(r'https://oss.xinchanjiao.com/upload/default/20220914-1bbc5a0e-9409-4ffa-adab-06f15090e2fb.xlsx')
#显示数据前5行
data.head()

In [ ]:
#方法1：使用for循环
for i in data['销售收入'].index:
    data.loc[i,'门店级别']='优秀门店' if data.loc[i,'销售收入']>=400000 else '非优秀门店'
data.head()

In [ ]:
#使用Series.map()
data['门店级别']=data['销售收入'].map(lambda x:'优秀门店' if x>=400000 else '非优秀门店')
data.head()

In [ ]:
#将数据中刚插入的门店级别‘列中‘优秀门店’替换成’A类门店，其他替换为‘其他门店
data['门店级别']=data['门店级别'].map({'优秀门店':'A类门店','非优秀门店':'其他门店'})
data

In [ ]:
#沿着0轴计算，销售收入、销售成本、销售毛利沿着数据列分别进行除以10000，保留四位小数
data[['销售收入','销售成本','销售毛利']]=data[['销售收入','销售成本','销售毛利']].apply(lambda x:round(x/10000,4),axis=0)
#修改列名
data.rename(columns={'销售收入':'销售收入(万元)','销售成本':'销售成本(万元)','销售毛利':'销售毛利(万元)'},inplace=True)
data.head()

In [ ]:
#按行(axis=1)的实现计算毛利率，计算结果后添加"%"，并存储在【销售毛利率】列
def func_01(row):
    row['销售毛利率']=format(row['销售毛利(万元)']/row['销售收入(万元)'],'2%')  #format()格式化函数
    return row
data[['销售收入(万元)','销售成本(万元)','销售毛利(万元)']].apply(func_01,axis=1)

In [ ]:
#将DataFrame中销售收入，销售成本和销售毛利统一保留一位小数显示
data[['销售收入(万元)','销售成本(万元)','销售毛利(万元)']].applymap(lambda x:"%.1f"%x)

## 4. 数据分组与聚合 (GroupBy & Aggregation)
- `groupby()`: 根据一个或多个键对数据进行分组。
- `sum()`, `mean()`: 对分组后的数据进行聚合计算。
- `agg()`: 同时应用多个聚合函数。

In [ ]:
#读取Excel文件
data=pd.read_excel(r'https://oss.xinchanjiao.com/upload/default/20220914-1bbc5a0e-9409-4ffa-adab-06f15090e2fb.xlsx')
#显示数据前5行
data.head()

In [ ]:
#创建一个英文月份和数字月份对照的字典，将英文月份转换为数字字符串
monthdict={'Jan':'01','Feb':'02','Mar':'03','Apr':'04','May':'05','Jun':'06',
'Jul':'07','Aug':'08','Sep':'09','Oct':'10','Nov':'11','Dec':'12'}
data['月']=data['月'].map(monthdict)
data

In [ ]:
#根据月列进行分组，求各月总和
data.groupby('月').sum()

In [ ]:
#将根据年、月列聚合，计算销售收入总和，销售成本平均值
data.groupby(['年','月']).agg({'销售收入':'sum','销售成本':'mean'})

In [ ]:
#读取Excel文件
data=pd.read_excel(r'https://oss.xinchanjiao.com/upload/default/20220914-1bbc5a0e-9409-4ffa-adab-06f15090e2fb.xlsx')
#显示数据前5行
data.head()

In [ ]:
#创建一个英文月份和数字月份对照的字典，将英文月份转换为数字字符串
monthdict={'Jan':'01','Feb':'02','Mar':'03','Apr':'04','May':'05','Jun':'06','Jul':'07','Aug':'08','Sep':'09','Oct':'10','Nov':'11','Dec':'12',}
data['月']= data['月'].map(monthdict)
data

In [ ]:
#行为月’，列为‘年，统计指标为销售收入的合计和销售毛利的平均数，在透视表中加上汇总栏
df1 =pd.pivot_table(data,
            index=['月'],
            columns=['年'],
            values=['销售收入','销售毛利'],
            aggfunc={'销售收入':'sum','销售毛利':'mean'},
            fill_value=0,
            margins=True,
            margins_name='汇总')
df1

## 5. 透视表与重塑 (Pivot Table & Reshaping)
- `pivot_table()`: 创建电子表格风格的透视表。
- `stack()`: 将列索引旋转为行索引（列转行）。
- `unstack()`: 将行索引旋转为列索引（行转列）。

In [ ]:
#根据月列进行分组，求各月总和
data1= data.groupby('月').sum()
data1

In [ ]:
#使用stack函数，将销售收入、销售成本、销售毛利索引转换到行索引
data1.stack(level=-1)

In [ ]:
#使用stack函数，将销售收入、销售成本、销售毛利索引转换到行索引
data1.stack().unstack(0)